# Replay an ITCH session through a cutoff

Build AAPL's order book from the beginning of a session through **14:00:21.321817 Eastern**, then inspect quotes, liquidity, price levels, cumulative depth, and a bounded sample of feed messages.

Run `make py-release` from the repository root to build the Python extension with Polars support, then select the project's Poetry Python kernel. The default input is `data/NASDAQ/01302020.NASDAQ_ITCH50`; set `LOBO_ITCH_PATH` and `LOBO_ITCH_SESSION_DATE` together to use another session.

1. Supply a timezone-aware cutoff and replay the original feed.
2. Inspect the resulting book through attributes and Polars lazy level views.
3. Inspect a small feed sample with `to_lazy_frame()` and `adapt()`.
4. Optionally compare the cutoff snapshot with a full-day replay using `None`.

In [ ]:
import os
from datetime import UTC, date, datetime, time
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from zoneinfo import ZoneInfo

import pandas as pd
import polars as pl
from IPython.display import display

from lobo.levels import levels_lazy
from lobo.replay import ReplayContext
from lobo.replay.adapters import itch

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "Cargo.toml").is_file() and (p / "python/lobo").is_dir()
)
DATA_PATH = Path(os.environ.get(
    "LOBO_ITCH_PATH", ROOT / "data/NASDAQ/01302020.NASDAQ_ITCH50"
)).expanduser().resolve()
SESSION_DATE = date.fromisoformat(os.environ.get("LOBO_ITCH_SESSION_DATE", "2020-01-30"))
TICKER = os.environ.get("LOBO_ITCH_TICKER", "AAPL")
EASTERN = ZoneInfo("America/New_York")
CUTOFF = datetime.combine(SESSION_DATE, time(14, 0, 21, 321817), tzinfo=EASTERN)
PREVIEW_MESSAGES = 500
TOP_LEVELS = 10

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Set LOBO_ITCH_PATH to your ITCH file: {DATA_PATH}")
if not hasattr(itch.ItchSource, "read_tickers"):
    raise RuntimeError("Rebuild with make py-release to enable Polars views.")

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(9)
{"file": str(DATA_PATH), "ticker": TICKER,
 "cutoff_eastern": CUTOFF.isoformat(), "same_instant_utc": CUTOFF.astimezone(UTC).isoformat()}

## Resolve the source and session time

The Python interface accepts an aware `datetime`, including `ZoneInfo`, UTC, or a fixed offset. The ITCH adapter uses the date to resolve EST/EDT, then compares local nanoseconds from midnight. The caller is responsible for supplying the session date. Python `datetime` has microsecond precision; native ITCH timestamps retain nanoseconds.

`ItchSource.all_available(str(DATA_PATH))` discovers every ticker when an all-book replay is wanted. This walkthrough reconstructs one ticker.

In [ ]:
source = itch.ItchSource(str(DATA_PATH), TICKER)
directory = itch.ItchSource.read_tickers(str(DATA_PATH))
selected_ticker = directory.filter(pl.col("ticker").str.to_uppercase() == TICKER.upper())
stock_locate = selected_ticker["stock_locate"].item()
display(selected_ticker)
source

## Replay from the beginning through the cutoff

The cutoff is **inclusive**. Messages after it do not update the book. The current replay implementation still scans the whole file, so a cutoff bounds the resulting state rather than the amount of input read.

Use a fresh context for each snapshot: an existing book cannot be rewound by lowering its cutoff. `collect_stats=True` makes `message_count` available here; the performance benchmark keeps statistics disabled and `cutoff_time=None`.

In [ ]:
started = perf_counter()
with ReplayContext(source=source, cutoff_time=CUTOFF, collect_stats=True) as replay:
    book = replay[TICKER]
    applied_messages = replay.message_count
    normalized_cutoff = replay.cutoff_time
elapsed_seconds = perf_counter() - started

assert normalized_cutoff == CUTOFF.astimezone(UTC)
assert applied_messages > 0, "Choose a cutoff after this ticker's first book event."
{"elapsed_seconds": round(elapsed_seconds, 3),
 "replay_messages_through_cutoff": applied_messages,
 "cutoff_utc": normalized_cutoff.isoformat(), "tickers": replay.keys()}

## Quotes and total liquidity

The returned book remains usable after leaving the context manager. ITCH prices are integer ticks with four decimal places: divide by 10,000 for display in dollars. Quantities are shares.

In [ ]:
best_bid = book.best_bid
best_ask = book.best_ask
quote = {
    "ticker": TICKER,
    "best_bid_usd": None if best_bid is None else best_bid / 10_000,
    "best_ask_usd": None if best_ask is None else best_ask / 10_000,
    "spread_usd": None if best_bid is None or best_ask is None else (best_ask - best_bid) / 10_000,
}
display(pl.DataFrame([quote]))

def liquidity_summary(book):
    return pl.DataFrame([
        {"side": name, "orders": side.order_count,
         "visible_shares": side.visible_quantity, "hidden_shares": side.hidden_quantity}
        for name, side in [("bid", book.orders.bids), ("ask", book.orders.asks)]
    ])

liquidity_summary(book)

## Lazy price-level views

`levels_lazy(...)` returns an ordinary Polars `LazyFrame`. These views read the book when collected. Sorting before `head()` makes the selected best-price levels explicit. `quantity` is visible quantity at a price; `order_count` counts resting orders at that price.

For Polars 1.43.2, disable predicate pushdown when collecting these level queries: its generated top-k predicate fails inside Python I/O sources. The queries still sort and select the requested levels.

In [ ]:
bids_lf = levels_lazy(book.orders.bids)
asks_lf = levels_lazy(book.orders.asks)
assert isinstance(bids_lf, pl.LazyFrame)
bids_lf.collect_schema()

LEVEL_QUERY_OPTIONS = pl.QueryOptFlags(predicate_pushdown=False)

def collect_levels(query):
    return query.collect(optimizations=LEVEL_QUERY_OPTIONS)

In [ ]:
def best_levels(levels, descending):
    return (
        levels.sort("price", descending=descending)
        .head(TOP_LEVELS)
        .with_columns((pl.col("price") / 10_000).alias("price_usd"))
        .select("price_usd", "quantity", "hidden_quantity", "order_count")
    )

print("Best bid levels")
display(collect_levels(best_levels(bids_lf, True)))
print("Best ask levels")
display(collect_levels(best_levels(asks_lf, False)))

## Cumulative depth and a combined price ladder

Cumulative visible shares run from the best price outward on each side. The bars below compare visible size among the displayed levels. This is the snapshot at the configured cutoff, not a time series.

In [ ]:
def depth_side(levels, side, descending):
    return (
        levels.sort("price", descending=descending)
        .with_columns(pl.col("quantity").cum_sum().alias("cumulative_shares"))
        .head(TOP_LEVELS)
        .with_columns(pl.lit(side).alias("side"), (pl.col("price") / 10_000).alias("price_usd"))
        .select("side", "price_usd", "quantity", "cumulative_shares", "order_count")
    )

depth_lf = pl.concat([depth_side(bids_lf, "bid", True), depth_side(asks_lf, "ask", False)])
depth = collect_levels(depth_lf)
assert depth.height <= TOP_LEVELS * 2
styled_depth = pd.DataFrame(depth.to_dicts()).style.format({
    "price_usd": "${:,.4f}", "quantity": "{:,.0f}",
    "cumulative_shares": "{:,.0f}", "order_count": "{:,.0f}",
}).hide(axis="index").bar(subset=["quantity"], color="#8bb9ca")
display(styled_depth)

# Descending price puts asks above bids in one compact ladder.
collect_levels(depth_lf.sort("price_usd", descending=True).select(
    "price_usd", "side", "quantity", "order_count"
))

In [ ]:
# Validate that the lazy level view agrees with the direct book attributes.
for name, levels, side in [
    ("bid", bids_lf, book.orders.bids), ("ask", asks_lf, book.orders.asks)
]:
    totals = collect_levels(levels.select(
        pl.len().alias("price_levels"), pl.col("quantity").sum().alias("visible_shares"),
        pl.col("order_count").sum().alias("orders"),
    )).row(0, named=True)
    assert totals["visible_shares"] == side.visible_quantity
    assert totals["orders"] == side.order_count
    print(name, totals)

## A bounded feed-message preview

Currently `source.to_lazy_frame()` decodes its complete input before returning a lazy frame; adding `.head()` afterward does not limit that initial read. To keep this example small, copy the first 500 messages for the selected stock locate into a temporary ITCH file, preserving their original bytes and directory record. The Rust adapter then decodes that small file.

This preview is an early-feed sample, not all events used to build the 14:00 snapshot. The actual book above was reconstructed from the original full session.

In [ ]:
def copy_ticker_preview(feed_path, output_path, locate, max_messages):
    copied = 0
    with feed_path.open("rb") as feed, output_path.open("wb") as output:
        while copied < max_messages:
            prefix = feed.read(2)
            if not prefix:
                break
            if len(prefix) != 2:
                raise ValueError("Truncated ITCH record length")
            length = int.from_bytes(prefix, "big")
            payload = feed.read(length)
            if len(payload) != length or length < 11:
                raise ValueError("Truncated ITCH message")
            if int.from_bytes(payload[1:3], "big") == locate:
                output.write(prefix + payload)
                copied += 1
    return copied

preview_directory = TemporaryDirectory(prefix="lobo-itch-preview-")
preview_path = Path(preview_directory.name) / "preview.itch"
preview_count = copy_ticker_preview(DATA_PATH, preview_path, stock_locate, PREVIEW_MESSAGES)
assert 0 < preview_count <= PREVIEW_MESSAGES
preview_source = itch.ItchSource(str(preview_path), TICKER)
feed_lf = preview_source.to_lazy_frame()
assert isinstance(feed_lf, pl.LazyFrame)
{"copied_messages": preview_count, "preview_bytes": preview_path.stat().st_size}

In [ ]:
feed_lf.select(
    "timestamp", "message_type", "stock", "operation", "reference", "shares", "price"
).head(12).collect()

## Filter and summarize messages lazily

`adapt()` selects message types that can update a replay book and returns a `ReplayMessages` object. Its `to_lazy_frame()` is a normal Polars lazy frame. These queries describe only the bounded preview; the context's `message_count` describes the full replay through the cutoff.

In [ ]:
replay_messages_lf = preview_source.adapt().to_lazy_frame()
cutoff_clock = CUTOFF.astimezone(EASTERN).time()
before_cutoff_lf = replay_messages_lf.filter(pl.col("timestamp") <= pl.lit(cutoff_clock))

display(before_cutoff_lf.select(
    "timestamp", "operation", "reference", "new_reference", "side", "shares", "price"
).head(12).collect())
replay_messages_lf.group_by("operation").len(name="sample_messages").sort(
    "sample_messages", descending=True
).collect()

## Try a full-day comparison

**Exercise:** enable the cell below and compare the end-of-file book with the cutoff snapshot. The source can be reused; the fresh context receives `cutoff_time=None` and applies the whole feed. This adds another complete replay.

A naive datetime, such as `datetime(2020, 1, 30, 14)`, is rejected. Supply `tzinfo=ZoneInfo("America/New_York")` or another explicit timezone. To inspect a different intermediate state, change `CUTOFF` and rerun from the replay cell with a new context.

In [ ]:
RUN_FULL_DAY_COMPARISON = False

if RUN_FULL_DAY_COMPARISON:
    with ReplayContext(source=source, cutoff_time=None, collect_stats=True) as full_day:
        closing_book = full_day[TICKER]
        assert full_day.message_count >= applied_messages
        print("Full-feed replay messages:", full_day.message_count)
        display(liquidity_summary(closing_book))
        display(collect_levels(best_levels(levels_lazy(closing_book.orders.bids), True)))
else:
    print("Set RUN_FULL_DAY_COMPARISON=True to replay the complete feed with cutoff_time=None.")

preview_directory.cleanup()